# Student Dropout & Academic Success — Model Training

**Dataset:** UCI *Predict Students' Dropout and Academic Success* (id 697)
**Task:** multi-class classification (Dropout / Enrolled / Graduate)

This notebook:
1. explores and cleans the data,
2. creates a stratified 80/20 train/test split (test split saved as `../test_data.csv`),
3. trains 5 classifiers, each as a full scikit-learn `Pipeline` (preprocessing + model),
4. evaluates them with the shared `utils.evaluate()` function (same code the Streamlit app uses),
5. saves the pipelines + metadata into this `model/` folder.

Run it from the `model/` folder (Jupyter does this automatically when you open the notebook here).

In [1]:
# ---- Imports -------------------------------------------------------------
import sys, json, time, platform
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# ---- Paths -----------------------------------------------------------------
# The notebook lives in model/, so the project root is one level up.
MODEL_DIR = Path.cwd() if Path.cwd().name == 'model' else Path.cwd() / 'model'
ROOT = MODEL_DIR.parent

# Make utils.py (in the project root) importable from this notebook
sys.path.insert(0, str(ROOT))
from utils import evaluate, clean_columns, METRIC_NAMES

RANDOM_STATE = 42          # fixed seed -> reproducible split and models
DATA_FILE = ROOT / 'data' / 'students_dropout.csv'
TARGET = 'Target'
print('Project root:', ROOT)

Project root: D:\Bits\BITS Sem 1\Assignment


## 1. Load and explore the data

In [2]:
# The UCI file is semicolon-separated and starts with a BOM -> use sep=';' and utf-8-sig
df = pd.read_csv(DATA_FILE, sep=';', encoding='utf-8-sig')
df = clean_columns(df)     # removes the stray tab in 'Daytime/evening attendance'

print('Shape (rows, columns):', df.shape)
print('Number of input features:', df.shape[1] - 1)
df.head()

Shape (rows, columns): (4424, 37)
Number of input features: 36


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [3]:
# Data types: all features are numeric (categorical ones are already integer-coded by UCI)
print(df.dtypes.value_counts(), '\n')
df.info()

int64      29
float64     7
str         1
Name: count, dtype: int64 

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-nu

In [4]:
# Missing values and duplicates
missing = df.isna().sum()
print('Columns with missing values:')
print(missing[missing > 0] if missing.any() else '  none')
print('\nDuplicate rows:', df.duplicated().sum())

Columns with missing values:
  none

Duplicate rows: 0


In [5]:
# Class balance: Enrolled is the minority class, so we use a STRATIFIED split
# and weighted-average metrics later.
counts = df[TARGET].value_counts()
balance = pd.DataFrame({'count': counts, 'percent': (counts / len(df) * 100).round(2)})
balance

,count,percent
Target,,
Graduate,2209,49.93
Dropout,1421,32.12
Enrolled,794,17.95


In [6]:
# Quick summary statistics of the numeric features
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Marital status,4424.0,1.18,0.61,1.00,1.00,1.00,1.00,6.00
Application mode,4424.0,18.67,17.48,1.00,1.00,17.00,39.00,57.00
Application order,4424.0,1.73,1.31,0.00,1.00,1.00,2.00,9.00
Course,4424.0,8856.64,2063.57,33.00,9085.00,9238.00,9556.00,9991.00
Daytime/evening attendance,4424.0,0.89,0.31,0.00,1.00,1.00,1.00,1.00
Previous qualification,4424.0,4.58,10.22,1.00,1.00,1.00,1.00,43.00
Previous qualification (grade),4424.0,132.61,13.19,95.00,125.00,133.10,140.00,190.00
Nacionality,4424.0,1.87,6.91,1.00,1.00,1.00,1.00,109.00
Mother's qualification,4424.0,19.56,15.60,1.00,2.00,19.00,37.00,44.00
Father's qualification,4424.0,22.28,15.34,1.00,3.00,19.00,37.00,44.00


## 2. Handle missing values and encode the target

* Rows without a target label are dropped (we cannot learn from them).
* Missing **feature** values are imputed *inside* the pipeline (median for numeric, most frequent for categorical).
  This way exactly the same rule is applied to new data in the Streamlit app.
* The text target is mapped to integers `0..n-1`; the mapping is stored in `metadata.json`.

In [7]:
df = df.dropna(subset=[TARGET]).reset_index(drop=True)

# Alphabetical order -> Dropout=0, Enrolled=1, Graduate=2
class_names = sorted(df[TARGET].astype(str).unique())
label_mapping = {name: idx for idx, name in enumerate(class_names)}
print('Label mapping:', label_mapping)

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str).map(label_mapping)

# Split the feature columns by type so each type gets the right preprocessing
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print(f'{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns')

Label mapping: {'Dropout': 0, 'Enrolled': 1, 'Graduate': 2}
36 numeric columns, 0 categorical columns


## 3. Stratified 80/20 train/test split
The test split is saved with the **original column names, unscaled, with the text target** so it can be uploaded straight into the Streamlit app.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

# Save the held-out test data (raw features + original text label)
test_df = X_test.copy()
test_df[TARGET] = df.loc[X_test.index, TARGET].values
test_df.to_csv(ROOT / 'test_data.csv', index=False)
print('Saved', ROOT / 'test_data.csv')

# Check that the class proportions are preserved in both splits
pd.DataFrame({'train %': y_train.value_counts(normalize=True).sort_index() * 100,
              'test %': y_test.value_counts(normalize=True).sort_index() * 100},
             ).rename(index=dict(enumerate(class_names))).round(2)

Train: (3539, 36)  Test: (885, 36)
Saved D:\Bits\BITS Sem 1\Assignment\test_data.csv


,train %,test %
Target,,
Dropout,32.13,32.09
Enrolled,17.94,17.97
Graduate,49.93,49.94


## 4. Preprocessing (ColumnTransformer)
* Numeric columns: median imputation, then `StandardScaler` (important for distance/margin based models: KNN, SVM, Logistic Regression).
* Categorical (text) columns: most-frequent imputation, then `OneHotEncoder` — only added if such columns exist (this dataset has none).

In [9]:
def make_one_hot_encoder():
    """OneHotEncoder with dense output that works on old and new scikit-learn.
    sklearn >= 1.2 calls the argument 'sparse_output', older versions call it 'sparse'."""
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def build_preprocessor():
    """Return a NEW (unfitted) preprocessor; every model pipeline gets its own copy."""
    transformers = [
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), numeric_cols)
    ]
    if categorical_cols:
        transformers.append(
            ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                              ('onehot', make_one_hot_encoder())]), categorical_cols)
        )
    return ColumnTransformer(transformers)

build_preprocessor()

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{fe

## 5. Define and train the 5 models
Each model is wrapped in a `Pipeline(preprocessing -> model)`, so the saved file contains *everything* needed to go from raw CSV values to a prediction.

In [10]:
# file name stem -> (display name, estimator)
models = {
    'logistic_regression': ('Logistic Regression',
                            LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    # max_depth limited so the tree does not memorise the training data
    'decision_tree': ('Decision Tree',
                      DecisionTreeClassifier(max_depth=6, min_samples_leaf=10,
                                             random_state=RANDOM_STATE)),
    # k=15 smooths the decision boundary compared to the default k=5
    'knn': ('KNN', KNeighborsClassifier(n_neighbors=15)),
    'naive_bayes': ('Naive Bayes', GaussianNB()),
    # probability=True is needed for predict_proba (AUC); RBF kernel = non-linear boundary
    'svm': ('Support Vector Machine',
            SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
}

pipelines, results, fit_report = {}, [], []

for key, (name, estimator) in models.items():
    pipe = Pipeline([('preprocess', build_preprocessor()), ('model', estimator)])

    start = time.time()
    pipe.fit(X_train, y_train)                     # learn ONLY from training data
    seconds = time.time() - start

    metrics, _, _ = evaluate(pipe, X_test, y_test) # score on unseen test data
    train_acc = accuracy_score(y_train, pipe.predict(X_train))

    pipelines[key] = pipe
    results.append({'ML Model Name': name, **metrics})
    fit_report.append({'ML Model Name': name, 'Train Accuracy': train_acc,
                       'Test Accuracy': metrics['Accuracy'],
                       'Gap (train - test)': train_acc - metrics['Accuracy'],
                       'Fit time (s)': seconds})
    print(f'{name:<24} trained in {seconds:5.1f}s  test accuracy = {metrics["Accuracy"]:.4f}')

Logistic Regression      trained in   0.3s  test accuracy = 0.7684
Decision Tree            trained in   0.1s  test accuracy = 0.7458


KNN                      trained in   0.0s  test accuracy = 0.7028
Naive Bayes              trained in   0.0s  test accuracy = 0.6588


D:\Bits\BITS Sem 1\Assignment\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Support Vector Machine   trained in   1.9s  test accuracy = 0.7582


## 6–7. Comparison table

In [11]:
results_df = pd.DataFrame(results).set_index('ML Model Name')[METRIC_NAMES].round(4)
results_df.to_csv(MODEL_DIR / 'metrics_comparison.csv')
print('Saved', MODEL_DIR / 'metrics_comparison.csv')
results_df

Saved D:\Bits\BITS Sem 1\Assignment\model\metrics_comparison.csv


,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.7684,0.8961,0.7500,0.7684,0.7531,0.6150
Decision Tree,0.7458,0.8687,0.7338,0.7458,0.7350,0.5780
KNN,0.7028,0.8368,0.6855,0.7028,0.6755,0.5019
Naive Bayes,0.6588,0.8118,0.6334,0.6588,0.6417,0.4279
Support Vector Machine,0.7582,0.8865,0.7465,0.7582,0.7454,0.5986


In [12]:
# Same table as Markdown -> copy-paste into README.md
def to_markdown_table(frame):
    header = '| ' + ' | '.join([frame.index.name] + list(frame.columns)) + ' |'
    divider = '|' + '---|' * (len(frame.columns) + 1)
    rows = ['| ' + ' | '.join([str(idx)] + [f'{v:.4f}' for v in row]) + ' |'
            for idx, row in frame.iterrows()]
    return '\n'.join([header, divider] + rows)

print(to_markdown_table(results_df))

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.7684 | 0.8961 | 0.7500 | 0.7684 | 0.7531 | 0.6150 |
| Decision Tree | 0.7458 | 0.8687 | 0.7338 | 0.7458 | 0.7350 | 0.5780 |
| KNN | 0.7028 | 0.8368 | 0.6855 | 0.7028 | 0.6755 | 0.5019 |
| Naive Bayes | 0.6588 | 0.8118 | 0.6334 | 0.6588 | 0.6417 | 0.4279 |
| Support Vector Machine | 0.7582 | 0.8865 | 0.7465 | 0.7582 | 0.7454 | 0.5986 |


In [13]:
# Train vs test accuracy: a large gap means overfitting
pd.DataFrame(fit_report).set_index('ML Model Name').round(4)

,Train Accuracy,Test Accuracy,Gap (train - test),Fit time (s)
ML Model Name,,,,
Logistic Regression,0.7731,0.7684,0.0047,0.3315
Decision Tree,0.7830,0.7458,0.0372,0.0551
KNN,0.7471,0.7028,0.0443,0.0160
Naive Bayes,0.6753,0.6588,0.0166,0.0110
Support Vector Machine,0.8302,0.7582,0.0720,1.8771


In [14]:
# Per-class report for each model (useful when writing the observations)
for key, (name, _) in models.items():
    print('=' * 60, '\n', name)
    print(classification_report(y_test, pipelines[key].predict(X_test),
                                target_names=class_names, digits=4, zero_division=0))

 Logistic Regression
              precision    recall  f1-score   support

     Dropout     0.7927    0.7676    0.7800       284
    Enrolled     0.5248    0.3333    0.4077       159
    Graduate     0.8035    0.9253    0.8601       442

    accuracy                         0.7684       885
   macro avg     0.7070    0.6754    0.6826       885
weighted avg     0.7500    0.7684    0.7531       885

 Decision Tree
              precision    recall  f1-score   support

     Dropout     0.7912    0.6937    0.7392       284
    Enrolled     0.4918    0.3774    0.4270       159
    Graduate     0.7840    0.9118    0.8431       442

    accuracy                         0.7458       885
   macro avg     0.6890    0.6609    0.6698       885
weighted avg     0.7338    0.7458    0.7350       885

 KNN
              precision    recall  f1-score   support

     Dropout     0.8036    0.6338    0.7087       284
    Enrolled     0.4400    0.2075    0.2821       159
    Graduate     0.6980    0.9253 

              precision    recall  f1-score   support

     Dropout     0.8408    0.7254    0.7788       284
    Enrolled     0.5000    0.3522    0.4133       159
    Graduate     0.7746    0.9253    0.8433       442

    accuracy                         0.7582       885
   macro avg     0.7051    0.6676    0.6785       885
weighted avg     0.7465    0.7582    0.7454       885



## 8. Save the pipelines and metadata

In [15]:
model_files = {}
for key, (name, _) in models.items():
    path = MODEL_DIR / f'{key}.joblib'
    joblib.dump(pipelines[key], path)
    model_files[name] = path.name
    print(f'Saved {path.name:<28} ({path.stat().st_size / 1024:.0f} KB)')

# Everything the Streamlit app needs to validate and encode an uploaded CSV
metadata = {
    'dataset': "UCI Predict Students' Dropout and Academic Success",
    'target': TARGET,
    'feature_columns': X.columns.tolist(),
    'numeric_columns': numeric_cols,
    'categorical_columns': categorical_cols,
    'class_names': class_names,
    'label_mapping': label_mapping,
    'model_files': model_files,
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'random_state': RANDOM_STATE,
    'sklearn_version': sklearn.__version__,
}
with open(MODEL_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved metadata.json')

Saved logistic_regression.joblib   (7 KB)
Saved decision_tree.joblib         (15 KB)
Saved knn.joblib                   (1029 KB)
Saved naive_bayes.joblib           (8 KB)
Saved svm.joblib                   (639 KB)
Saved metadata.json


## 9. Library versions
Pin **scikit-learn** and **numpy** in `requirements.txt` to these versions so Streamlit Cloud can load the saved `.joblib` files.

In [16]:
print('Python      ', platform.python_version())
print('scikit-learn', sklearn.__version__)
print('numpy       ', np.__version__)
print('pandas      ', pd.__version__)
print('joblib      ', joblib.__version__)

Python       3.12.8
scikit-learn 1.9.1
numpy        2.5.4
pandas       3.1.1
joblib       1.6.0
